# BANKING77: frozen DistilBERT vs. LoRA

Run in Google Colab: **Runtime ? Change runtime type ? T4 GPU** (or another available GPU).
Upload `data/colab/banking77_colab_inputs.zip` when prompted. Run cells in order.
No account token or paid API is needed for the public base model.

We compare a frozen encoder with a trained classification head against the same encoder
with LoRA and a trained head. Both use the same initial head weights, split, seed,
learning rate, batch size, and five-epoch budget. Select each best epoch by validation macro-F1.
This is an initial controlled comparison, not independent hyperparameter tuning.
LoRA is not guaranteed to beat the frozen model or the 84.79% TF-IDF baseline.
The official test set is excluded. No urgency model or review threshold is trained here.

Colab storage is temporary. Download each experiment's ZIP immediately after it finishes.
If the runtime is lost, rerun setup and training; optimizer-state resume is not implemented.


In [ ]:
%pip install -q --upgrade --only-binary=:all: transformers==4.57.6 tokenizers==0.22.2 peft==0.18.1 accelerate==1.12.0 "huggingface-hub>=0.34,<1" "safetensors>=0.6.2,<1" "scikit-learn>=1.6,<2"


After installation succeeds, restart the Colab session (Runtime > Restart session), then continue from the next cell. This clears any older imported packages. The notebook keeps Colab's GPU-enabled PyTorch. Binary-only installation prevents an unexpected Rust build of tokenizers.


In [ ]:
import csv, gc, hashlib, io, json, platform, shutil, time, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import transformers, peft, sklearn, tokenizers, accelerate, huggingface_hub, safetensors
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, set_seed
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from huggingface_hub import HfApi
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from google.colab import files
from tqdm.auto import tqdm

assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
DEVICE = torch.device('cuda')
SEED, EPOCHS, BATCH_SIZE, MAX_LENGTH, LR = 42, 5, 16, 128, 3e-4
MODEL_ID = 'distilbert/distilbert-base-uncased'
REVISION = HfApi().model_info(MODEL_ID).sha
ROOT = Path('/content/banking77_experiment')
ROOT.mkdir(exist_ok=True)
VERSIONS = dict(python=platform.python_version(), torch=torch.__version__,
                transformers=transformers.__version__, peft=peft.__version__, sklearn=sklearn.__version__,
                tokenizers=tokenizers.__version__, accelerate=accelerate.__version__,
                huggingface_hub=huggingface_hub.__version__, safetensors=safetensors.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('Versions:', VERSIONS)
print('Base model revision:', REVISION)


## Upload and verify the existing splits


In [ ]:
uploaded = files.upload()
assert len(uploaded) == 1, 'Upload only banking77_colab_inputs.zip.'
expected = {'train.csv', 'validation.csv', 'label_mapping.json', 'baseline_metrics.json'}
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    assert len(archive.namelist()) == 4 and set(archive.namelist()) == expected
    contents = {name: archive.read(name) for name in expected}
baseline = json.loads(contents['baseline_metrics.json'])
label2id = json.loads(contents['label_mapping.json'])
assert len(label2id) == 77 and sorted(label2id.values()) == list(range(77))
id2label = {value: key for key, value in label2id.items()}
assert hashlib.sha256(contents['label_mapping.json']).hexdigest() == baseline['label_mapping_sha256']
frames = {}
for split in ['train', 'validation']:
    assert hashlib.sha256(contents[f'{split}.csv']).hexdigest() == baseline['input_sha256'][split]
    frame = pd.read_csv(io.BytesIO(contents[f'{split}.csv']), keep_default_na=False)
    assert frame['text'].str.strip().ne('').all()
    assert set(frame['intent']) == set(label2id)
    assert (frame['intent'].map(label2id) == frame['label']).all()
    frames[split] = frame
normalize = lambda text: ' '.join(text.lower().split())
assert not set(frames['train']['text'].map(normalize)) & set(frames['validation']['text'].map(normalize))
print({split: len(frame) for split, frame in frames.items()})
print('Verified identical inputs to the local baseline. Test data is absent.')
del uploaded


## Tokenization and model setup
The two head layers (`pre_classifier` and `classifier`) are trained in both experiments. LoRA additionally adapts the attention query/value layers with rank 8. Other encoder weights stay frozen.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
class Tickets(Dataset):
    def __init__(self, frame):
        self.encoded = tokenizer(frame['text'].tolist(), truncation=True, max_length=MAX_LENGTH)
        self.labels = frame['label'].tolist()
    def __len__(self): return len(self.labels)
    def __getitem__(self, index):
        return {**{key: value[index] for key, value in self.encoded.items()}, 'labels': self.labels[index]}
train_data, val_data = Tickets(frames['train']), Tickets(frames['validation'])
collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)
# Report truncation instead of silently hiding it.
for name, frame in frames.items():
    lengths = tokenizer(frame['text'].tolist(), truncation=False, return_length=True)['length']
    print(name, 'queries longer than MAX_LENGTH:', sum(n > MAX_LENGTH for n in lengths))

def fresh_base():
    set_seed(SEED)
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID, revision=REVISION, num_labels=77, id2label=id2label,
        label2id=label2id, use_safetensors=True)

def make_model(kind):
    model = fresh_base()
    if kind == 'frozen':
        for parameter in model.distilbert.parameters(): parameter.requires_grad = False
    elif kind == 'lora':
        model = get_peft_model(model, LoraConfig(
            task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16, lora_dropout=0.05,
            target_modules=['q_lin', 'v_lin'],
            modules_to_save=['pre_classifier', 'classifier'], bias='none'))
    else: raise ValueError(kind)
    names = [name for name, p in model.named_parameters() if p.requires_grad]
    assert names and all('classifier' in name or (kind == 'lora' and 'lora_' in name) for name in names)
    if kind == 'lora': assert any('lora_' in name for name in names)
    print(kind, 'trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))
    return model

@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    scores, truth = [], []
    for batch in loader:
        truth.extend(batch.pop('labels').tolist())
        logits = model(**{k: v.to(DEVICE) for k, v in batch.items()}).logits
        scores.append(logits.softmax(-1).cpu().numpy())
    scores = np.concatenate(scores)
    predictions = scores.argmax(-1)
    metrics = dict(accuracy=float(accuracy_score(truth, predictions)),
                   macro_f1=float(f1_score(truth, predictions, labels=list(range(77)), average='macro', zero_division=0)))
    return metrics, np.asarray(truth), predictions, scores


## Training function
This uses an explicit PyTorch loop in float32 for clarity and stability. Batch size 16 and
128-token inputs keep memory use modest. Each epoch evaluates validation data; only the best
checkpoint is retained. Downloaded LoRA checkpoints include **both trained head layers**.
Training time below includes epoch validation and checkpoint saving, but excludes model download.
CPU baseline and GPU training times are not directly comparable hardware benchmarks.


In [ ]:
def run_experiment(kind):
    model = make_model(kind).to(DEVICE)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator,
                        generator=torch.Generator().manual_seed(SEED))
    val_loader = DataLoader(val_data, batch_size=BATCH_SIZE * 2, collate_fn=collator)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.01)
    folder = ROOT / kind
    folder.mkdir(exist_ok=True)
    checkpoint = folder / 'model'
    best, best_epoch, history = -1.0, None, []
    torch.cuda.synchronize()
    start = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        loss_sum = 0.0
        for batch in tqdm(loader, desc=f'{kind} epoch {epoch}/{EPOCHS}'):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            loss = model(**batch).loss
            if not torch.isfinite(loss): raise RuntimeError('Non-finite training loss')
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            loss_sum += loss.item() * len(batch['labels'])
        metrics, _, _, _ = evaluate(model, val_loader)
        history.append(dict(epoch=epoch, training_loss=loss_sum / len(train_data), **metrics))
        print(history[-1])
        if metrics['macro_f1'] > best:
            best, best_epoch = metrics['macro_f1'], epoch
            model.save_pretrained(checkpoint, safe_serialization=True)
            tokenizer.save_pretrained(checkpoint)
        (folder / 'history.json').write_text(json.dumps(history, indent=2))
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    del optimizer, model
    gc.collect(); torch.cuda.empty_cache()
    # Reload the saved best checkpoint for the reported metrics and inference.
    if kind == 'lora':
        model = PeftModel.from_pretrained(fresh_base(), checkpoint).to(DEVICE)
    else:
        model = AutoModelForSequenceClassification.from_pretrained(checkpoint, use_safetensors=True).to(DEVICE)
    metrics, truth, predicted, scores = evaluate(model, val_loader)
    assert abs(metrics['macro_f1'] - best) < 1e-6, 'Best checkpoint did not reload correctly'
    report = dict(experiment=kind, **metrics, best_epoch=best_epoch, history=history,
                  training_seconds=elapsed, trainable_parameters=trainable, total_parameters=total,
                  base_model=MODEL_ID, base_revision=REVISION, seed=SEED, epochs=EPOCHS,
                  learning_rate=LR, batch_size=BATCH_SIZE, max_length=MAX_LENGTH,
                  weight_decay=0.01, lora=(dict(rank=8, alpha=16, dropout=0.05, targets=['q_lin','v_lin']) if kind == 'lora' else None),
                  input_sha256=baseline['input_sha256'], label_mapping_sha256=baseline['label_mapping_sha256'],
                  evaluation_split='validation', test_set_used=False, versions=VERSIONS,
                  gpu=torch.cuda.get_device_name(0), saved_best_model_reload_check=True,
                  classification_report=classification_report(truth, predicted, labels=list(range(77)),
                      target_names=[id2label[i] for i in range(77)], output_dict=True, zero_division=0))
    (folder / 'metrics.json').write_text(json.dumps(report, indent=2))
    (folder / 'label_mapping.json').write_text(json.dumps(label2id, indent=2))
    pd.DataFrame(dict(text=frames['validation']['text'], actual=[id2label[i] for i in truth],
                      predicted=[id2label[i] for i in predicted], confidence=scores.max(-1))).to_csv(folder / 'validation_predictions.csv', index=False)
    pd.DataFrame(confusion_matrix(truth, predicted, labels=list(range(77))),
                 index=[id2label[i] for i in range(77)], columns=[id2label[i] for i in range(77)]).to_csv(folder / 'confusion_matrix.csv')
    archive = shutil.make_archive(str(ROOT / f'{kind}_artifacts'), 'zip', folder)
    print('Saved:', archive)
    del model
    gc.collect(); torch.cuda.empty_cache()
    return report, archive


## Experiment 1: frozen encoder
The pretrained encoder weights stay fixed. Only the two classification-head layers learn.


In [ ]:
frozen_report, frozen_zip = run_experiment('frozen')
files.download(frozen_zip)


## Experiment 2: LoRA
Start again from the original pretrained weights and the same seeded initial head, not from Experiment 1.


In [ ]:
lora_report, lora_zip = run_experiment('lora')
files.download(lora_zip)


## Compare validation results
These are initial single-seed results. Do not claim an improvement unless measurements support it. Tune both baselines fairly on validation data before the final held-out test evaluation. Confidence is uncalibrated; review-threshold selection is a later step.


In [ ]:
comparison = pd.DataFrame([
    dict(model='TF-IDF + Logistic Regression', accuracy=baseline['accuracy'], macro_f1=baseline['macro_f1'],
         trainable_parameters=None, training_seconds=baseline['training_seconds'], hardware='local CPU'),
    *[dict(model=r['experiment'], accuracy=r['accuracy'], macro_f1=r['macro_f1'],
           trainable_parameters=r['trainable_parameters'], training_seconds=r['training_seconds'], hardware=r['gpu'])
      for r in [frozen_report, lora_report]]])
display(comparison)
print('LoRA minus TF-IDF accuracy (percentage points):', round(100 * (lora_report['accuracy'] - baseline['accuracy']), 2))
print('LoRA minus frozen macro-F1:', round(lora_report['macro_f1'] - frozen_report['macro_f1'], 4))
comparison.to_csv(ROOT / 'comparison.csv', index=False)
files.download(str(ROOT / 'comparison.csv'))


## Try a message with the saved LoRA model
This demonstrates inference only. An unfamiliar or ambiguous message may still get a confident prediction.


In [ ]:
checkpoint = ROOT / 'lora' / 'model'
model = PeftModel.from_pretrained(fresh_base(), checkpoint).to(DEVICE).eval()
message = "I sent money yesterday but my friend still hasn't received it."
inputs = tokenizer(message, return_tensors='pt', truncation=True, max_length=MAX_LENGTH).to(DEVICE)
with torch.inference_mode():
    probabilities = model(**inputs).logits.softmax(-1)[0]
values, indices = probabilities.topk(3)
for value, index in zip(values.tolist(), indices.tolist()):
    print(id2label[index], round(value, 4))
del model
gc.collect(); torch.cuda.empty_cache()


## What to keep
Save `frozen_artifacts.zip`, `lora_artifacts.zip`, and `comparison.csv` on your laptop.
The frozen ZIP contains the full model. The LoRA ZIP contains adapters and both head layers;
it also needs the base model at `base_revision` from `metrics.json`. Restore its 77-label mapping
when constructing the base model, as shown above. The tokenizer is included in each checkpoint.
Bring back the comparison and metrics before selecting a model or using the official test set.

Data attribution: PolyAI, Casanueva et al. (2020), *Efficient Intent Detection with Dual Sentence Encoders*, CC BY 4.0.
[Dataset](https://github.com/PolyAI-LDN/task-specific-datasets) ?
[PEFT LoRA configuration](https://huggingface.co/docs/peft/v0.18.1/en/package_reference/lora).
